# Geometric metrics on two real collapsed domains

This replaces the synthetic two-diamond example. The earlier toy notebook is archived with the output snapshot. Every geometric metric uses **current physical reach attributes** and a validated before/after domain. No routing or recalibration is performed.

Cases: **02 / S013_unit_20**, a simple two-reach unit; **46 / CB006_bubble_101**, a small compound with 11 reaches and five complete routes. Both children have one replacement reach. This compound is a real interconnected unit, not two separate diamonds joined by arbitrary centreline segments.

In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path('/Users/6256481/Code/river-hierarchy');OUT=Path('/Users/6256481/Code/river-hierarchy/RAPID/outputs/real_geometry_examples_20261012')
sys.path.insert(0,str(ROOT/'RAPID/src'))
from rapid_tools.current_review import parent_child_network_figure
cases=json.loads((OUT/'cases.json').read_text())
values=pd.read_csv(OUT/'metric_values.csv')
def table(case,state,kind):return pd.read_csv(OUT/f"{cases[case]['tag']}_{state}_{kind}.csv")
display(pd.DataFrame(cases).T[['example','state_id','source','sink','parent_links','child_links','domain_type']])
def metric_comparison(metric):
    shown=values[values.metric.eq(metric)].pivot(index='case',columns='state',values='value').reindex(cases)
    if metric in {'upstream_distance','downstream_distance','network_entry_exposure','modified_length_fraction'}:
        return shown[['parent']].rename(columns={'parent':'parent_context'})
    for state in ['parent','child']:
        if state not in shown:shown[state]=np.nan
    if metric=='route_length_quantiles':return shown[['parent','child']]
    shown=shown.apply(pd.to_numeric,errors='coerce')
    if metric in {'route_length_skew','log1p_paths','dominance_mean','dominance_sd','split_entropy_mean','split_entropy_sd','split_exposure_mean','split_exposure_sd'}:
        shown['child_minus_parent']=shown.child-shown.parent
    else:
        valid=shown.parent.gt(0)&shown.child.ge(0)
        shown['child_over_parent']=(shown.child/shown.parent.where(valid)).where(valid)
        shown['relative_change_pct']=100*(shown.child_over_parent-1)
    return shown


/Users/6256481/.matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /var/folders/bs/xx1t3fn91jl3k26fn1pndd_c0000gn/T/matplotlib-f0cz_sax because there was an issue with the default path (/Users/6256481/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
Matplotlib is building the font cache; this may take a moment.


,example,state_id,source,sink,parent_links,child_links,domain_type
Simple,02,S013_unit_20,40,39,"[58, 59]",[79],closed_unit
Small compound,46,CB006_bubble_101,25,23,"[34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45]",[541],closed_compound_unit


## Notation: reach, route and domain are different objects

- $e$ identifies **one physical reach/link**. $L_e$ is that reach's length in metres; $W_e$ (or $w_e$) is its measured/derived hydrographic width in metres. **$L_e$ is not a sum of path lengths.**
- $k$ identifies **one complete source-to-sink route**. Its length is $L_k=\sum_{e\in k}L_e$.
- $\sum_eL_e$ sums each unique reach in the chosen domain once. It is different from both a route length and mean travelled distance.
- $p_{u,e}=W_e/\sum_{j\in out(u)}W_j$ is a local outgoing allocation fraction. It is a width proxy, not observed discharge.
- $P_k$ is the probability of one complete route, obtained by multiplying its local split fractions.
- $r_u$ and $r_e$ sum the allocation arriving at a node or passing through a reach. They include all contributing route histories.

Width columns in the reach tables refer to the same hydrographic widths used by current routing preparation. Routing subdivisions are not treated as new physical reaches.

## The domain answer

**For the local collapse predictors here, yes: the metrics are calculated for the collapsed comparison domain, before and after the collapse, with matched physical inlet/outlet boundaries.** Simple: nodes 40→39, parent links 58/59 and child link 79. Compound: nodes 25→23, parent 11 links and child link 541.

The first foundation dataset instead computed whole-network summaries. Those remain valid **network-context** quantities, but are not local-domain measurements. This notebook now supplies real local-domain examples explicitly; it does not claim that the full 2,172-case local feature table has been rebuilt.

For a member edit inside a larger compound, distinguish the edited closed domain from the containing compound before/after. If the edit is not closed, a larger enclosing matched domain may be necessary and must be labelled. For these two full-unit collapses, the validated comparison domain is the collapsed unit itself.

Exceptions are clearly contextual: upstream/downstream distances, entry exposure, and modified-length fraction need the whole parent network as reference. Their values below describe the parent domain's position/exposure, not intrinsic width or length within the domain.

## Simple: actual parent and child geometry

The map is zoomed to the validated collapse domain; surrounding reaches are grey. The following reach and route tables are the numerical inputs for all worked metric values.

In [2]:
case='Simple'
record=pd.read_json(OUT/(cases[case]['tag']+'_record.json'),typ='series')
record['example_id']=str(record['example_id']).zfill(2)
parent_child_network_figure(record,zoom_to_domain=True).show()
for state in ['parent','child']:
    print(state)
    display(table(case,state,'edges').round(6))
    display(table(case,state,'routes').round(6))

parent
child


<ipython-input-2-7177095a4beb>:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  record=pd.read_json(OUT/(cases[case]['tag']+'_record.json'),typ='series')
<ipython-input-2-7177095a4beb>:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  record=pd.read_json(OUT/(cases[case]['tag']+'_record.json'),typ='series')
<ipython-input-2-7177095a4beb>:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as date

,id_link,u,v,L_e_m,W_e_m,p_outgoing,r_edge,mean_length_contribution_m
0,58,40,39,1721.277918,390.718814,0.880891,0.880891,1516.257794
1,59,40,39,2016.964750,52.830871,0.119109,0.119109,240.239160


,route,P_k,L_k_m,W_harmonic_k_m
0,58,0.880891,1721.277918,390.718814
1,59,0.119109,2016.964750,52.830871


,id_link,u,v,L_e_m,W_e_m,p_outgoing,r_edge,mean_length_contribution_m
0,79,40,39,1700.890008,449.563031,1.0,1.0,1700.890008


,route,P_k,L_k_m,W_harmonic_k_m
0,79,1.0,1700.890008,449.563031


## Small compound: actual parent and child geometry

The map is zoomed to the validated collapse domain; surrounding reaches are grey. The following reach and route tables are the numerical inputs for all worked metric values.

In [3]:
case='Small compound'
record=pd.read_json(OUT/(cases[case]['tag']+'_record.json'),typ='series')
record['example_id']=str(record['example_id']).zfill(2)
parent_child_network_figure(record,zoom_to_domain=True).show()
for state in ['parent','child']:
    print(state)
    display(table(case,state,'edges').round(6))
    display(table(case,state,'routes').round(6))

parent
child


<ipython-input-3-0d3c61873818>:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  record=pd.read_json(OUT/(cases[case]['tag']+'_record.json'),typ='series')
<ipython-input-3-0d3c61873818>:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as datetime strings, matching the behavior without a 'unit'. To retain the old behavior, explicitly cast ints or floats to numeric type before calling to_datetime.
  record=pd.read_json(OUT/(cases[case]['tag']+'_record.json'),typ='series')
<ipython-input-3-0d3c61873818>:2: FutureWarning: The behavior of 'to_datetime' with 'unit' when parsing strings is deprecated. In a future version, strings will be parsed as date

,id_link,u,v,L_e_m,W_e_m,p_outgoing,r_edge,mean_length_contribution_m
0,34,25,23,6700.599979,134.567537,0.182762,0.182762,1224.612314
1,45,25,31,138.175617,601.733436,0.817238,0.817238,112.922421
2,42,31,28,1672.321712,135.974668,0.113876,0.093064,155.632131
3,43,31,29,78.744168,1058.088602,0.886124,0.724175,57.024550
4,38,29,27,1467.419000,619.312338,0.694587,0.503002,738.115098
5,41,29,30,931.505059,272.314737,0.305413,0.221173,206.023415
6,39,30,27,779.029732,143.017911,0.519382,0.114873,89.489620
7,40,30,28,28.016254,132.343553,0.480618,0.106299,2.978112
8,36,27,26,369.103529,704.357433,1.000000,0.617875,228.060011
9,37,28,26,1058.986088,183.551771,1.000000,0.199363,211.122596


,route,P_k,L_k_m,W_harmonic_k_m
0,45 → 43 → 38 → 36 → 35,0.503002,4536.038149,605.052120
1,45 → 43 → 41 → 40 → 37 → 35,0.106299,4718.023020,336.874689
2,45 → 43 → 41 → 39 → 36 → 35,0.114873,4779.153939,341.483367
3,45 → 42 → 37 → 35,0.093064,5352.079251,236.846561
4,34,0.182762,6700.599979,134.567537


,id_link,u,v,L_e_m,W_e_m,p_outgoing,r_edge,mean_length_contribution_m
0,541,25,23,4693.697291,918.774803,1.0,1.0,4693.697291


,route,P_k,L_k_m,W_harmonic_k_m
0,541,1.0,4693.697291,918.774803


## Confluences restore allocation; complete-route probabilities do not reset

The recursion is
$$r_v=\sum_{e:u\to v}r_up_{u,e},\qquad r_e=r_up_{u,e}.$$
Every incoming contribution is **added** before outgoing allocation is applied. For a 90:10 split that fully rejoins, the downstream node receives $0.9+0.1=1$. Repeating such splits does not reduce the allocation on a common downstream reach to $0.9^n$. That product describes only the particular route that chose the 90% branch every time.

In the real compound, node 26 receives contributions from nodes 27 and 28. Their sum restores its allocation to about 0.817238, equal to the share that entered the indirect branch at node 25. At the final confluence, node 23 also receives the direct branch (link 34), bringing the total to **1.0**.

There is no probability renormalization trick: all path histories are included. A confluence with an additional outgoing split accumulates first, then redistributes the sum.

In [4]:
nodes=table('Small compound','parent','nodes');edges=table('Small compound','parent','edges')
display(nodes.round(6))
display(edges[edges.v.isin([26,23])][['id_link','u','v','p_outgoing','r_edge']].round(6))
for node in [26,23]:
    incoming=edges.loc[edges.v.eq(node),'r_edge'].sum()
    expected=nodes.loc[nodes.node.eq(node),'visit_probability'].iloc[0]
    print(f'Node {node}: sum incoming = {incoming:.9f}; node allocation = {expected:.9f}')
    assert np.isclose(incoming,expected)
assert np.isclose(nodes.loc[nodes.node.eq(23),'visit_probability'].iloc[0],1.)

Node 26: sum incoming = 0.817238409; node allocation = 0.817238409
Node 23: sum incoming = 1.000000000; node allocation = 1.000000000


,node,in_degree,out_degree,visit_probability
0,25,0,2,1.000000
1,31,1,2,0.817238
2,29,1,2,0.724175
3,30,1,2,0.221173
4,27,2,1,0.617875
5,28,2,1,0.199363
6,26,2,1,0.817238
7,23,2,0,1.000000


,id_link,u,v,p_outgoing,r_edge
0,34,25,23,0.182762,0.182762
8,36,27,26,1.000000,0.617875
9,37,28,26,1.000000,0.199363
10,35,26,23,1.000000,0.817238


## Repeated shared reaches: why expected length is different from unweighted summation

$$\mu_L=\sum_kP_kL_k=\sum_er_eL_e.$$
Shared reaches may appear in several route descriptions; that is correct. Their contributions are weighted by the route probabilities and add to their total allocation $r_e$. A reach traversed by all routes has $r_e=1$, not a multiplicity equal to the number of routes. Summing complete-path widths as if all routes were independent parallel channels lacks this property.

The next check computes each real mean independently from the explicit route list and the edge allocations. Production graphs use DAG recursions; enumeration is only used here because these examples have two and five routes.

In [5]:
for case in cases:
    rt=table(case,'parent','routes');ed=table(case,'parent','edges')
    a=(rt.P_k*rt.L_k_m).sum();b=ed.mean_length_contribution_m.sum()
    print(case, 'route probabilities sum =',rt.P_k.sum(),'mean from routes =',a,'m; from reaches =',b,'m')
    assert np.isclose(a,b) and np.isclose(rt.P_k.sum(),1.)

Simple route probabilities sum = 0.9999999999999999 mean from routes = 1756.4969546281209 m; from reaches = 1756.496954628121 m
Small compound route probabilities sum = 0.9999999999999999 mean from routes = 5054.852937067474 m; from reaches = 5054.852937067475 m


## Reading the comparisons: ratios first, parent structure retained

For a nonnegative metric with positive parent value, show $R_X=X_c/X_p$ and $100(R_X-1)$. Thus 0.8 means the child retains 80%, a 20% decrease. Positive quantities may optionally use $\ln R_X$, but no log is defined when the child is zero. Never add an arbitrary epsilon to create one.

**A ratio alone is insufficient for complete collapses:** bifurcation count 1→0 and 4→0 both give ratio zero; both route SDs also collapse to zero. Retain parent magnitude to distinguish the removed structure. The tables retain parent and child values. Signed skewness uses a difference; conditional split means remain undefined for a no-split child. Context metrics are parent-only.

Start with mean travelled distance ($\mu_L$), route spread, width size/spread, one split-balance descriptor, bifurcation count/expected split encounters, domain exposure/position, and the explicitly parameterized resistance ratio. Deprioritize width/route quantiles and bifurcation density; do not select algebraically redundant topology measures together. This is a discussion shortlist, not a fitted selection result.

Use dominance **or** entropic effective branches initially. They describe related balance information; normalized entropy and inverse-Simpson count are additional alternatives, not automatically additional predictors.


Bounded split-balance measures use differences when both states are defined; log path count uses a difference of logs. Ratios are emphasized for size, spread and resistance, not applied indiscriminately to every transformed metric.

## Each metric: real values, formula and interpretation

All lengths are metres; area/variance terms are square metres; CV, entropy, counts, allocation and elongation are dimensionless; bifurcation density is per metre. Each table shows parent, child and change where defined. The child has one route, so route-length SD/CV are zero. A mean conditional on encountering a split is **undefined** when there are no splits; those cells are NaN, rather than falsely implying zero dominance or zero branch width.

Route skewness is the standard third standardized central moment $E[(L-\mu_L)^3]/\sigma_L^3$, not an invented exponent. It is undefined for a single deterministic route. [NIST definition](https://www.itl.nist.gov/div898/software/dataplot/refman2/ch2/skewness.pdf).

### total_length

**Family:** length. **Equation:** $$\sum_e L_e$$ unique reaches.

Sum of unique physical reach lengths

Audit correlations within and across networks; no automatic univariate exclusion.

In [6]:
metric='total_length'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,1700.890008,3738.242668,0.454997,-54.500278
Small compound,4693.697291,15706.496972,0.298838,-70.116205


### area_proxy

**Family:** width. **Equation:** $$\sum_e L_e*w_e$$.

Sum of length times width; channel-area proxy, not polygon union

Audit correlations within and across networks; no automatic univariate exclusion.

In [7]:
metric='area_proxy'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,7.646573e+05,7.790937e+05,0.981470,-1.852974
Small compound,4.312451e+06,4.460988e+06,0.966703,-3.329703


### width_mean

**Family:** width. **Equation:** $$\bar{w}=\frac{\sum_e L_ew_e}{\sum_e L_e}$$.

Length-weighted reach width mean

Sum L_e W_e divided by sum L_e over unique physical reaches. This differs from flow-weighted width.

In [8]:
metric='width_mean'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,449.563031,208.411743,2.157091,115.709069
Small compound,918.774803,284.021858,3.234874,223.487358


### width_sd

**Family:** width. **Equation:** $$\sigma_w=\sqrt\frac{\sum_e L_e(w_e-\bar{w})^2}{\sum_e L_e}$$`.

Length-weighted reach width standard deviation

Audit correlations within and across networks; no automatic univariate exclusion.

In [9]:
metric='width_sd'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.000000e+00,168.414646,0.000000e+00,-100.0
Small compound,1.136868e-13,213.851753,5.316152e-16,-100.0


### width_cv

**Family:** width. **Equation:** $$CV_w =\sigma_{w}/\bar{w}$$.

Length-weighted width SD/mean

Audit correlations within and across networks; no automatic univariate exclusion.

In [10]:
metric='width_cv'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.000000e+00,0.808086,0.000000e+00,-100.0
Small compound,1.237374e-16,0.752941,1.643388e-16,-100.0


### width_harmonic

**Family:** width. **Equation:** $$\frac{\sum_e L_e}{\sum_e(\frac{L_e}{w_e})}$$.

Total length / sum(length/width)

Sum L_e divided by sum L_e/W_e. Narrow reaches matter strongly. This aggregate is not equivalent nonlinear network resistance.

In [11]:
metric='width_harmonic'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,449.563031,87.786833,5.121076,412.107588
Small compound,918.774803,186.011803,4.939336,393.933604


### width_p10

**Family:** width. **Equation:** `inverse length-weighted width CDF at 0.1`.

Length-weighted width 10th percentile

Audit correlations within and across networks; no automatic univariate exclusion.

**Priority: exploratory only.** No clear added mechanism beyond width spread/constriction measures has been established. Keep out of the initial shortlist. For a two-reach unit these quantiles often reduce to choosing one reach width.

In [12]:
metric='width_p10'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,449.563031,52.830871,8.509476,750.947610
Small compound,918.774803,134.567537,6.827611,582.761104


### width_p90

**Family:** width. **Equation:** `inverse length-weighted width CDF at 0.9`.

Length-weighted width 90th percentile

Audit correlations within and across networks; no automatic univariate exclusion.

**Priority: exploratory only**, for the same reason as width P10. This is not a finding of zero predictive value; it is a choice to start with more interpretable summaries.

In [13]:
metric='width_p90'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,449.563031,390.718814,1.150605,15.060502
Small compound,918.774803,619.312338,1.483540,48.354029


### bifurcations

**Family:** topological complexity. **Equation:** `count(vertices with out-degree > 1)`.

Number of physical branching vertices; not routing cells

Audit correlations within and across networks; no automatic univariate exclusion.

In [14]:
metric='bifurcations'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,1.0,0.0,-100.0
Small compound,0.0,4.0,0.0,-100.0


### cycle_rank

**Family:** topological complexity. **Equation:** `E-V+C (undirected multigraph)`.

E-V+C on the underlying undirected multigraph

E−V+C counts undirected independent loops. Parallel physical reaches are separate edges. The flow graph remains directed and acyclic.

**Priority: redundancy diagnostic.** On these connected single-sink DAGs it equals junction excess exactly; do not select both.

In [15]:
metric='cycle_rank'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,1.0,0.0,-100.0
Small compound,0.0,4.0,0.0,-100.0


### junction_excess

**Family:** topological complexity. **Equation:** `sum_v max(out_degree(v)-1,0)`.

Sum(max(out_degree-1,0))

Sum max(outdegree−1,0). Equals cycle rank for these closed single-sink graphs.

**Priority: redundancy diagnostic, not a separate initial predictor.** In the 15 base graphs, 1,740/1,746 branching nodes are binary (99.66%). For binary splits, junction excess equals bifurcation count. It differs only at the few higher-degree split nodes.

In [16]:
metric='junction_excess'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,1.0,0.0,-100.0
Small compound,0.0,4.0,0.0,-100.0


### bifurcation_density

**Family:** topological complexity. **Equation:** `bifurcations/total_length`.

Bifurcations per metre of unique physical channel

Audit correlations within and across networks; no automatic univariate exclusion.

**Priority: secondary.** Simple = 0.2675 splits/km of unique channel; compound = 0.2547 splits/km. Similar values indicate similar splitting per channel length, although the counts are 1 and 4. Two examples cannot establish predictive usefulness; start with count and physical length separately. Density is their ratio, not an independent ingredient.

In [17]:
metric='bifurcation_density'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,0.000268,0.0,-100.0
Small compound,0.0,0.000255,0.0,-100.0


### route_length_mean

**Family:** length. **Equation:** $$\mu_L=\sum_e r_e*L_e = \sum_{k} P_{k}*L_{k}$$.

Expected complete route length under width/reference split allocation

For the simple parent, multiply the two printed reach lengths by their outgoing width fractions. For the compound, use all five P_k values, or the r_e L_e contributions. This is not sum of unique lengths.

**This is exactly $\mu_L$.** `route_length_mean` is the code name for mean travelled distance, not another metric.

In [18]:
metric='route_length_mean'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,1700.890008,1756.496955,0.968342,-3.165787
Small compound,4693.697291,5054.852937,0.928553,-7.144731


### route_length_sd

**Family:** length. **Equation:** $$\sigma_L=\sqrt{\sum_k P_k(L_k-\mu_L)^2}$$.

SD of allocated route length

Spread between complete allocated routes, not spread among individual link lengths. Complete collapse to one route removes this geometric spread.

In [19]:
metric='route_length_sd'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,95.777998,0.0,-100.0
Small compound,0.0,812.042136,0.0,-100.0


### route_length_cv

**Family:** length. **Equation:** $$CV_L = \frac{\sigma_L}{\mu_{L}}$$.

Allocated route length SD/mean

Audit correlations within and across networks; no automatic univariate exclusion.

In [20]:
metric='route_length_cv'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,0.054528,0.0,-100.0
Small compound,0.0,0.160646,0.0,-100.0


### route_length_min

**Family:** length. **Equation:**$$L_{min} = \min(L_K)=\min(\sum_e L_e)$$.

Shortest directed source-to-sink physical length

Audit correlations within and across networks; no automatic univariate exclusion.

In [21]:
metric='route_length_min'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,1700.890008,1721.277918,0.988155,-1.184464
Small compound,4693.697291,4536.038149,1.034757,3.475701


### route_length_max

**Family:** length. **Equation:** $$L_{max} = \max(L_K)=\max(\sum_e L_e)$$.

Longest directed source-to-sink physical length

Audit correlations within and across networks; no automatic univariate exclusion.

In [22]:
metric='route_length_max'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,1700.890008,2016.964750,0.843292,-15.670811
Small compound,4693.697291,6700.599979,0.700489,-29.951089


### route_length_skew

**Family:** length. **Equation:** $$\sum_k P_k*(L_k-\mu_L)^3 / \sigma_L^3$$.

Third central route-length moment divided by SD cubed; zero for deterministic routes

Unstable for near-zero SD; deterministic zero convention requires a flag.

In [23]:
metric='route_length_skew'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,NaN,2.351780,NaN
Small compound,NaN,1.382683,NaN


### log1p_paths

**Family:** topological complexity. **Equation:** `ln(1+number_of_directed_source_to_sink_paths)`.

Natural log(1+complete directed path count); explicit complexity only

The simple parent has two complete routes: ln(3). The compound has five: ln(6). A single child route gives ln(2), not zero.

In [24]:
metric='log1p_paths'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,0.693147,1.098612,-0.405465
Small compound,0.693147,1.791759,-1.098612


## Split metrics in one small table

Read the columns in this order: **how much reaches the split** ($r_b$), **how it divides** ($p_{bj}$), **largest share** (dominance), **effective branch count** ($\exp H_b$). These are different descriptions of the same splits, not four unrelated mechanisms.

The real compound has four split nodes, but an expected 2.7626 encounters per allocated route: not every route visits every bifurcation. The visit-weighted means divide by $\sum_b r_b$; route entropy instead sums the contributions without that normalization.


In [25]:
splits=pd.read_csv(OUT/'split_explanation.csv')
display(splits.round(5))

,case,bifurcation,r_b,outgoing_fractions,dominance,local_entropy,normalized_entropy,entropic_effective_branches,inverse_simpson_branches,route_entropy_contribution,split_impact
0,Simple,40,1.00000,0.88089 / 0.11911,0.88089,0.36515,0.52680,1.44073,1.26557,0.36515,0.20984
1,Small compound,25,1.00000,0.18276 / 0.81724,0.81724,0.47556,0.68608,1.60891,1.42596,0.47556,0.29872
2,Small compound,29,0.72417,0.69459 / 0.30541,0.69459,0.61538,0.88781,1.85036,1.73693,0.44564,0.30725
3,Small compound,30,0.22117,0.51938 / 0.48062,0.51938,0.69240,0.99892,1.99850,1.99700,0.15314,0.11042
4,Small compound,31,0.81724,0.11388 / 0.88612,0.88612,0.35454,0.51150,1.42553,1.25284,0.28975,0.16493


### route_entropy

**Family:** width. **Equation:** $$\sum_b r_bH_b; H_b=-\sum_j p_{bj}\ln(p_{bj})$$.

Expected accumulated -log branch probability; not a bounded local imbalance

Audit correlations within and across networks; no automatic univariate exclusion.

**Plain interpretation:** total uncertainty about which complete route an allocated particle follows. A split that is almost always visited contributes more than one rarely visited.

$$H_{route}=\sum_b r_bH_b=-\sum_kP_k\ln P_k.$$

$r_b$ is the fraction reaching bifurcation $b$, conditional on entering the unit (confluences already included). $H_b=-\sum_jp_{bj}\ln p_{bj}$ describes balance at that split. Local split entropy measures balance; route entropy combines balance **and the number of split encounters**. Keep route entropy as a secondary diagnostic initially, since this combination is less direct to interpret.

In [26]:
metric='route_entropy'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,0.365146,0.0,-100.0
Small compound,0.0,1.364084,0.0,-100.0


### expected_splits

**Family:** topological complexity. **Equation:** `sum_b r_b`.

Expected number of branching vertices encountered by a particle

Audit correlations within and across networks; no automatic univariate exclusion.

In [27]:
metric='expected_splits'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,1.000000,0.0,-100.0
Small compound,0.0,2.762586,0.0,-100.0


### dominance_mean

**Family:** width. **Equation:** `visit-weighted mean of max_j p_bj`.

Visit-weighted mean largest outgoing fraction at branching vertices; zero if none

Audit correlations within and across networks; no automatic univariate exclusion.

At one split, dominance is the largest outgoing share: 90:10 gives 0.9; 50:50 gives 0.5.

$$\bar d=\frac{\sum_b r_b\max_jp_{bj}}{\sum_b r_b}.$$

This is the average largest outgoing share at an encountered bifurcation. Simple unit: **0.88089** (88.1:11.9 at its only split). Small compound: **0.78162**, a visit-weighted average over four split nodes. It is not the probability of one dominant complete route. For a child with no split it is undefined, rather than zero or a forced value of one.

In [28]:
metric='dominance_mean'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,NaN,0.880891,NaN
Small compound,NaN,0.781619,NaN


### dominance_sd

**Family:** width. **Equation:** `visit-weighted SD of max_j p_bj`.

Visit-weighted SD of largest outgoing fraction; zero if none

Audit correlations within and across networks; no automatic univariate exclusion.

In [29]:
metric='dominance_sd'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,NaN,0.000000,NaN
Small compound,NaN,0.105741,NaN


### effective_branches_mean

**Family:** width. **Equation:** `visit-weighted mean of 1/sum_j p_bj^2`.

Visit-weighted mean 1/sum(p_j squared) at branching vertices

Audit correlations within and across networks; no automatic univariate exclusion.

In [30]:
metric='effective_branches_mean'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,1.265574,NaN,NaN
Small compound,NaN,1.501984,NaN,NaN


### effective_branches_sd

**Family:** width. **Equation:** `visit-weighted SD of 1/sum_j p_bj^2`.

Visit-weighted SD effective branch count

Audit correlations within and across networks; no automatic univariate exclusion.

In [31]:
metric='effective_branches_sd'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,0.000000,NaN,NaN
Small compound,NaN,0.233542,NaN,NaN


### split_entropy_mean
**Family:** width. **Equation:** visit-weighted mean of $$\frac{H_b}{ln(k_b)}$$.


Visit-weighted mean local entropy / log(out_degree)

Audit correlations within and across networks; no automatic univariate exclusion.

In [32]:
metric='split_entropy_mean'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,NaN,0.526795,NaN
Small compound,NaN,0.712360,NaN


### split_entropy_sd

**Family:** width. **Equation:** `visit-weighted SD of H_b/ln(k_b)`.

Visit-weighted SD normalized local split entropy

Audit correlations within and across networks; no automatic univariate exclusion.

In [33]:
metric='split_entropy_sd'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,NaN,0.000000,NaN
Small compound,NaN,0.163794,NaN


### split_exposure_mean

**Family:** width. **Equation:** `mean_b r_b`.

Unweighted mean conditional node visit probability across bifurcations

Conditional visit probability averaged over split nodes. Confluences have already accumulated all incoming contributions.

In [34]:
metric='split_exposure_mean'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,NaN,1.000000,NaN
Small compound,NaN,0.690646,NaN


### split_exposure_sd

**Family:** width. **Equation:** `SD_b r_b`.

SD conditional node visit probability across bifurcations

Audit correlations within and across networks; no automatic univariate exclusion.

In [35]:
metric='split_exposure_sd'
display(metric_comparison(metric))

state,child,parent,child_minus_parent
case,,,
Simple,NaN,0.000000,NaN
Small compound,NaN,0.288641,NaN


### split_impact_mean

**Family:** width. **Equation:** `mean_b I_b; $$I_b=r_b*(1-\sum_j p_{bj}^2)$$.

Mean across bifurcations of visit_probability*(1-sum(p_j squared))

Audit correlations within and across networks; no automatic univariate exclusion.

In [36]:
metric='split_impact_mean'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,0.209844,NaN,NaN
Small compound,NaN,0.220330,NaN,NaN


### split_impact_sd

**Family:** width. **Equation:** `SD_b I_b`.

SD across bifurcations of visit_probability*(1-sum(p_j squared))

Audit correlations within and across networks; no automatic univariate exclusion.

In [37]:
metric='split_impact_sd'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,0.000000,NaN,NaN
Small compound,NaN,0.084924,NaN,NaN


### split_impact_sum

**Family:** width. **Equation:** `sum_b I_b`.

Sum of visit_probability*(1-sum(p_j squared)); repeated splitting can exceed one

Equals bifurcation count times split impact mean.

In [38]:
metric='split_impact_sum'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,0.209844,0.0,-100.0
Small compound,0.0,0.881319,0.0,-100.0


### branch_length_contrast_mean

**Family:** length. **Equation:** `sum_b r_b*s_b / sum_b r_b; s_b^2=sum_j p_bj*(L_bj+mu_downstream_j-mu_b)^2`.

Visit-weighted mean SD among branch expected remaining distances

At each split, compare expected remaining distance through each outgoing branch to the same domain sink; weight local contrasts by node visits.

In [39]:
metric='branch_length_contrast_mean'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,95.777998,NaN,NaN
Small compound,NaN,380.519445,NaN,NaN


### branch_length_contrast_sd

**Family:** length. **Equation:** `sqrt(sum_b r_b*(s_b-mean_s)^2 / sum_b r_b)`.

Visit-weighted SD of branch distance contrast

Audit correlations within and across networks; no automatic univariate exclusion.

In [40]:
metric='branch_length_contrast_sd'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,0.000000,NaN,NaN
Small compound,NaN,306.429209,NaN,NaN


### branch_length_variance_sum

**Family:** length. **Equation:** `sum_b r_b*s_b^2 = route_length_sd^2`.

Sum visit_probability*Var(branch expected remaining distance); equals route-length variance

Equals route-length SD squared. This is a redundant alternative representation, not another independent predictor.

In [41]:
metric='branch_length_variance_sum'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,0.0,9173.424941,0.0,-100.0
Small compound,0.0,659412.430211,0.0,-100.0


### area_elongation

**Family:** length. **Equation:** `mu_L^2/sum_e L_e*w_e`.

Allocated mean route length squared / unique-edge area proxy; hypothesis only

Candidate only; not equivalent to obsolete sum-of-complete-path widths.

In [42]:
metric='area_elongation'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,3.783429,3.960091,0.955390,-4.461039
Small compound,5.108648,5.727775,0.891908,-10.809202


### sinuosity_proxy

**Family:** length. **Equation:** `mu_L/L_min (not map sinuosity)`.

Allocated mean route length / shortest directed route; not map sinuosity

Audit correlations within and across networks; no automatic univariate exclusion.

In [43]:
metric='sinuosity_proxy'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,1.0,1.020461,0.979949,-2.005072
Small compound,1.0,1.114376,0.897363,-10.263697


### entropic_branches_mean

**Family:** width. **Equation:** `visit-weighted mean of exp(H_b)`.

Proposed local eBI analogue; not published cross-section eBI. Compare against inverse-Simpson count, not both by default.

Translate local entropy back to an effective number of branches: $N_{1,b}=\exp(H_b)$. A 50:50 split behaves like 2 equally used branches; a 90:10 split like 1.384. Values near one indicate strong dominance.

$$\bar N_1=\frac{\sum_b r_b\exp(H_b)}{\sum_b r_b}.$$

Simple parent: **1.44073**; small compound parent: **1.64914**. These are averages of effective branches **at split encounters**, not total route counts. With no child bifurcations, there is no encounter to average over: NaN is appropriate.

This uses the effective-number transformation underlying [the entropic braiding index](https://agupubs.onlinelibrary.wiley.com/doi/abs/10.1029/2022GL099681). Published eBI uses channel fractions at cross-sections. We apply it at bifurcations and average over visits, so this is an **adaptation**, not a measured cross-sectional eBI. Do not call exponentiated route entropy a simultaneous channel count.

In [44]:
metric='entropic_branches_mean'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,1.440725,NaN,NaN
Small compound,NaN,1.649144,NaN,NaN


### entropic_branches_sd

**Family:** width. **Equation:** `visit-weighted SD of exp(H_b)`.

No-split undefined, with has_splits flag; do not interpret zero as balance.

No-split undefined, with has_splits flag; do not interpret zero as balance.

Spread of those local effective branch counts: do the bifurcations all have similar balance, or do some strongly favour one branch while others divide evenly? This is optional variability around the mean, not another mean braiding index.

In [45]:
metric='entropic_branches_sd'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,NaN,0.000000,NaN,NaN
Small compound,NaN,0.189112,NaN,NaN


### path_harmonic_width_min

**Family:** width. **Equation:** `min_path [sum L_e / sum(L_e/w_e)]`.

Narrowest effective serial path; does not measure route probability.

Minimum of the listed real route harmonic widths. An extreme path width is not an allocation fraction.

Among the listed complete routes, choose the smallest harmonic effective width. It identifies the narrowest effective route, even if that route carries little allocation. Keep as a secondary constriction diagnostic.

In [46]:
metric='path_harmonic_width_min'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,449.563031,52.830871,8.509476,750.947610
Small compound,918.774803,134.567537,6.827611,582.761104


### path_harmonic_width_max

**Family:** width. **Equation:** `max_path [sum L_e / sum(L_e/w_e)]`.

Widest effective serial path; never sum over all complete paths.

Maximum of the listed real route harmonic widths. Never add these overlapping path widths as independent channel capacities.

Among the listed complete routes, choose the largest harmonic effective width. It identifies the widest effective route, not necessarily the most probable route. Compare its added information against simpler width summaries.

In [47]:
metric='path_harmonic_width_max'
display(metric_comparison(metric))

state,child,parent,child_over_parent,relative_change_pct
case,,,,
Simple,449.563031,390.718814,1.150605,15.060502
Small compound,918.774803,605.052120,1.518505,51.850522


### route_length_quantiles

**Family:** length. **Equation:** `weighted quantiles of route length under P_path`.

Prefer weighted tails to unweighted extrema; large graphs need an audited distribution algorithm.

Inverse discrete CDF of the listed route probabilities. Values are P10/P50/P90; there is no interpolation between distinct route lengths.

**Priority: exploratory only.** With two or five discrete routes, P10/P90 can mainly pick individual paths. Mean and SD are the first route-distribution candidates to review.

In [48]:
metric='route_length_quantiles'
display(metric_comparison(metric))

state,parent,child
case,,
Simple,1721.278 / 1721.278 / 2016.965 m (P10/P50/P90),1700.890 / 1700.890 / 1700.890 m (P10/P50/P90)
Small compound,4536.038 / 4536.038 / 6700.600 m (P10/P50/P90),4693.697 / 4693.697 / 4693.697 m (P10/P50/P90)


### upstream_distance

**Family:** length. **Equation:** `conditional expected inlet-to-domain-entry geometric distance`.

Separate edit-domain and containing-compound boundaries.

Separate edit-domain and containing-compound boundaries.

Parent-context variable only: how far the allocated flow travels from the network inlet before reaching this domain. No child/parent ratio is shown because this teaching table has only the parent context.

In [49]:
metric='upstream_distance'
display(metric_comparison(metric))

state,parent_context
case,
Simple,12828.273920600079
Small compound,758685.9802275667


### downstream_distance

**Family:** length. **Equation:** `expected domain-exit-to-network-outlet geometric distance`.

For full-network collapse external distances are zero.

For full-network collapse external distances are zero.

Parent-context variable only: expected remaining distance after leaving this domain to the network outlet. This is not a distance inside the unit.

In [50]:
metric='downstream_distance'
display(metric_comparison(metric))

state,parent_context
case,
Simple,105106.75973972154
Small compound,46623.79611594295


### network_entry_exposure

**Family:** width. **Equation:** `network inlet probability reaching selected domain source`.

Global fraction, distinct from conditional split exposure inside domain.

Whole-network allocation arriving at the domain inlet. Within-domain calculations reset entry allocation to one, so this separate factor is needed for network exposure.

Parent-context variable only: fraction of the whole-network allocation reaching the domain inlet. In contrast, r_b starts from one at the local domain inlet.

In [51]:
metric='network_entry_exposure'
display(metric_comparison(metric))

state,parent_context
case,
Simple,1.0
Small compound,0.9999999999999993


### modified_length_fraction

**Family:** length. **Equation:** `unique modified parent length / unique whole-parent length`.

Physical edited geometry must not be replaced by containing-compound length.

For these complete unit collapses, sum of unique parent-domain reach lengths divided by unique whole-parent network length. Not a sum of path lengths.

Parent-context variable only: unique length removed/modified by this full unit collapse divided by unique parent-network length. This is already a fraction; do not divide by a nonexistent child version.

In [52]:
metric='modified_length_fraction'
display(metric_comparison(metric))

state,parent_context
case,
Simple,0.0235505121679253
Small compound,0.011508177264744051


## Resistance restored: matched-domain ratio under the stated geometry law

For a wide section, $A_e=W_eh_e$, $R_{h,e}\approx h_e$. The repository's Chezy resistance is

$$K_e=\frac{L_e}{C_z^2g\,W_e^2h_e^3}.$$

With $h_e=aW_e^{\alpha}$ and a common $a,C_z$ in parent and child,

$$K_e=\frac{1}{C_z^2ga^3}\frac{L_e}{W_e^{2+3\alpha}}.$$

The common prefactor scales the entire network's equivalent resistance and therefore cancels in $K_c/K_p$. For the current $\alpha=0.65$, the width power is **3.95**. Both real examples verify a common $a$ across their parent and child reaches to numerical precision.

We can therefore calculate a **geometry-derived resistance ratio conditional on this fixed depth–width/Chezy law** using normalized edge weights

$$\widetilde K_e=\frac{L_e/L_0}{(W_e/W_0)^{3.95}},\qquad R_K=\frac{\widetilde K_c}{\widetilde K_p},\qquad resistance\_log=-\log_{10}R_K.$$

$L_0,W_0$ are the same on both sides. The static unit-head network calculation includes series/parallel topology and internal confluence continuity; it does **not** force the width-proxy split probabilities used by route metrics. Resistance is thus not a harmonic-width substitute. No hydrograph, calibrated Manning n, or celerity enters this geometry ratio. The exponent and wide-section law are explicit modeling assumptions and require sensitivity assessment.

The normalized-geometry result was checked against a calculation using the saved current depths and $C_z=22$; ratios agree at relative tolerance $10^{-8}$ and continuity residuals are below $10^{-8}$. This is a static equivalent-resistance calculation, not a RAPID reroute or recalibration. No production files were changed.

Archived selected-domain ratios are printed separately for provenance. They differ from the current-depth-law calculation and must not be mixed into the new table as if they were the same version.


**Existing resistance reproduced:** the stored ratios are also reproduced independently with a constant width/depth ratio, $h=aW$ ($\alpha=1$, width power 5), using the same current domain lengths and widths. The table below therefore distinguishes the existing-law ratio from the current-depth-law ratio. Neither is silently substituted for the other. Choice of exponent remains an explicit predictor assumption.

In [53]:
resistance=pd.read_csv(OUT/'resistance_current_geometry.csv').set_index('case')
comparison=pd.DataFrame({
    'existing_law_Kc_over_Kp (alpha=1)':resistance.constant_aspect_ratio,
    'current_depth_law_Kc_over_Kp (alpha=.65)':resistance.child_over_parent,
    'existing_resistance_log':-np.log10(resistance.constant_aspect_ratio),
    'current_depth_resistance_log':resistance.resistance_log,
})
display(comparison)
np.testing.assert_allclose(resistance.constant_aspect_ratio,resistance.archived_ratio_reference_only,rtol=1e-8)


,existing_law_Kc_over_Kp (alpha=1),current_depth_law_Kc_over_Kp (alpha=.65),existing_resistance_log,current_depth_resistance_log
case,,,,
Simple,0.496103,0.588104,0.304428,0.230546
Small compound,0.132691,0.227356,0.877157,0.643294


The metrics above remain local-domain examples, not a rebuilt full-population local feature dataset. Parent-only context metrics remain separate. The earlier whole-network snapshot is unchanged. User-edited mathematical formulas elsewhere in this notebook have been retained.